<h1 style = "background-color: #00FFFF; color :black;", align='center'><br> Financial Distress Prediction <br></h1>


<img src="https://i0.wp.com/bardwellcreative.com/wp-content/uploads/2019/01/Financial-Distress-banner.jpg?fit=888%2C259&ssl=1" alt="Financial-Distress-banner.webp">

## Introduction
Financial distress prediction is a key issue in measuring corporate solvency. Its main objective is to distinguish normal companies from those at risk of financial distress. For enterprises themselves, financial distress prediction can help to identify risks early, make plans according to the actual situation, and adjust business strategy. For investors, financial distress prediction can help determine the financial risks of enterprises and choose investment projects reasonably according to their risk preferences. For regulators, financial distress prediction can help to understand the financial status of each company in a timely manner, do a good job in supervision and management, and maintain the stability of the financial market. Therefore, how to predict the financial distress of enterprises effectively has become a hot topic in academic and business circles.

## Problem Statement
We have been given four data files that include train data, test data, sample submissions and data dictinary. In the data dictionary, meaning of each atttribute from the train and test data is given. By exploring the attributes, we have to find/predict whether somebody will experience financial distress in the next two years.
The goal of this project, is to build a model that borrowers can use to help make the best financial decisions. And for that we will train our model with large amount of data ('the train data' where labels already given) and use hyper parameter tuning to increase it's accuracy. At the end, we will make predictions on test data and save them along with our model.

<div class="alert alert-block alert-info"> <h3>📌Column Description :</h3> <br>
    <p style='color:black;'>
        <b>SeriousDlqin2yrs</b> : 90 days past due delinquency or worse<br>
        <b>RevolvingUtilizationOfUnsecuredLines</b> : Total balance on credit cards and personal lines of credit except real estate and no installment debt like car loans divided by the sum of credit limits <br>
        <b>age</b> : Age of borrower in years<br>
        <b>NumberOfTime30-59DaysPastDueNotWorse</b> : Number of times borrower has been 30-59 days past due but no worse in the last 2 years.<br>
        <b>DebtRatio</b> : Monthly debt payments, alimony,living costs divided by monthy gross income<br>
        <b>MonthlyIncome</b> : Monthly income<br>
        <b>NumberOfOpenCreditLinesAndLoans</b> : Number of Open loans (installment like car loan or mortgage) and Lines of credit (e.g. credit cards)<br>
        <b>NumberOfTimes90DaysLate</b> : Number of times borrower has been 90 days or more past due.<br>
        <b>NumberRealEstateLoansOrLines</b> : Number of mortgage and real estate loans including home equity lines of credit<br>
        <b>NumberOfTime60-89DaysPastDueNotWorse</b> : Number of times borrower has been 60-89 days past due but no worse in the last 2 years.<br>
        <b>NumberOfDependents</b> : Number of dependents in family excluding themselves (spouse, children etc.)
    </p>
</div>

## Importing Libraries

In [ ]:
#ignoring warnings to keep the code clean
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#importing dependencies
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib
import os
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

## Loading Data

In [ ]:
train_df = pd.read_csv('/kaggle/input/GiveMeSomeCredit/cs-training.csv')
print(train_df.shape)
train_df.head()

In [ ]:
test_df = pd.read_csv('/kaggle/input/GiveMeSomeCredit/cs-test.csv')
print(test_df.shape)
test_df.head()

In [ ]:
submission =  pd.read_csv('/kaggle/input/GiveMeSomeCredit/sampleEntry.csv')
submission.head()

In [ ]:
train_df['SeriousDlqin2yrs'].unique()

Labels are in the form of binary numbers.

In [ ]:
train_df.info()

In [ ]:
test_df.info()

## Exploratory Data Analysis

### Stastical Information

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

In [ ]:
train_df['SeriousDlqin2yrs'].value_counts()

Observe that the number of `0's` are lot more than the `1's`, training with such distribution had lead me to lesser accuracy therefore, we will only take fraction of data whose label is `0` this time.

In [ ]:
train0=train_df[train_df['SeriousDlqin2yrs']==0].sample(frac=0.06684)
train1=train_df[train_df['SeriousDlqin2yrs']==1].copy()
train_df=pd.concat([train0, train1], axis=0)
train_df['SeriousDlqin2yrs'].value_counts()

### Distribution of data

In [ ]:
Atttributes= ['RevolvingUtilizationOfUnsecuredLines', 'age',
              'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
              'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
              'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
              'NumberOfDependents']

In [ ]:
for i in Atttributes:
    fig, axes = plt.subplots(1,2, figsize=(15, 5))
    axes[0].set_title(i+' (Train Data)')
    sns.distplot(train_df[i], ax=axes[0])
    axes[1].set_title(i+' (Test Data)')
    sns.distplot(test_df[i], ax=axes[1])

There are lot of outliers in both dataset. We need to take care of them properly so that they don't trouble us after scaling the data. For that we will substitute the outliers with value which comes under two times stadard deviation. We choosing two times standard deviation since it covers 95% of the part of the distribution.

In [ ]:
def normalizer(x,df):
    upper_boundary=df[x].mean()+2*df[x].std()
    lower_boundary=df[x].mean()-2*df[x].std()
    max_att=df[x].max()
    min_att=df[x].min()
    return {'Attribute':x, 'upper_boundary': upper_boundary, 'lower_boundary': lower_boundary, 
           'max_att':max_att, 'min_att':min_att }

In [ ]:
train_limits = pd.DataFrame([normalizer(x, train_df) for x in Atttributes])
train_limits

In [ ]:
test_limits = pd.DataFrame([normalizer(x, test_df) for x in Atttributes])
test_limits

In [ ]:
def NormAtt(i, lim_df, df):
    Att=lim_df.iloc[i].Attribute
    UL=lim_df.iloc[i].upper_boundary
    LL=lim_df.iloc[i].lower_boundary
    fig, axes = plt.subplots(1,2, figsize=(15, 5))
    axes[0].set_title('Old Distribution of '+Att)
    sns.distplot(df[Att], ax=axes[0])
    df.loc[df[Att]<LL,Att]=LL
    df.loc[df[Att]>UL,Att]=UL
    axes[1].set_title('New Distribution of '+Att)
    sns.distplot(df[Att], ax=axes[1])

In [ ]:
for i in range(0,10):
    NormAtt(i, train_limits, train_df)

In [ ]:
for i in range(0,10):
    NormAtt(i, test_limits, test_df)

#### Correlation

In [ ]:
sns.heatmap(train_df.corr())
plt.title('Correlation Between Attributes');

In train data `NumberOfTime60-89DaysPastDueNotWorse` shows good correlation with `NumberOfTime30-59DaysPastDueNotWorse` and `NumberOfTimes90DaysLate`.

In [ ]:
sns.heatmap(test_df.corr())
plt.title('Correlation Between Attributes');

Similiarly, in test data `NumberOfTime60-89DaysPastDueNotWorse` shows good correlation with `NumberOfTime30-59DaysPastDueNotWorse` and `NumberOfTimes90DaysLate`.

### Input and Target Columns

In [ ]:
train_df.columns

In [ ]:
input_cols = ['RevolvingUtilizationOfUnsecuredLines', 'age',
       'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
       'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       'NumberOfDependents']
target_col = 'SeriousDlqin2yrs'

In [ ]:
inputs = train_df[input_cols].copy()
targets = train_df[target_col].copy()

In [ ]:
test_inputs = test_df[input_cols].copy()

### Imputing missing numeric values

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy = 'median').fit(train_df[input_cols])

In [ ]:
inputs[input_cols] = imputer.transform(inputs[input_cols])
test_inputs[input_cols] = imputer.transform(test_inputs[input_cols])

In [ ]:
inputs[input_cols].isna().sum()

In [ ]:
test_inputs[input_cols].isna().sum()

### Scaling Numeric Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler().fit(train_df[input_cols])

In [ ]:
inputs[input_cols] = scaler.transform(inputs[input_cols])
test_inputs[input_cols] = scaler.transform(test_inputs[input_cols])

In [ ]:
inputs.describe().loc[['min', 'max']]

In [ ]:
test_inputs.describe().loc[['min', 'max']]

In [ ]:
inputs.head()

In [ ]:
test_inputs.head()

### Modeling : DecisionTreeClassifier

#### Splitting

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, train_targets, val_targets = train_test_split(inputs, targets, test_size=0.25)

#### Training

We can use `DecisionTreeClassifier` from `sklearn.tree` to train a decision tree.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model = DecisionTreeClassifier(random_state=42)

In [ ]:
#fitting the model
model.fit(X_train, train_targets)

#### Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
train_preds = model.predict(X_train)

In [ ]:
train_preds

In [ ]:
pd.value_counts(train_preds)

In [ ]:
#Probabilities for each prediction
train_probs = model.predict_proba(X_train)

In [ ]:
train_probs

In [ ]:
accuracy_score(train_targets, train_preds)

The training set accuracy is close to 100%! But we can't rely solely on the training set accuracy, we must evaluate the model on the validation set too.We can make predictions and compute accuracy in one step using `model.score`

In [ ]:
model.score(X_val, val_targets)

It appears that the model has learned the training examples perfect, and doesn't generalize well to previously unseen examples. This phenomenon is called "overfitting", and reducing overfitting is one of the most important parts of any machine learning project.

In [ ]:
val_targets.value_counts() / len(val_targets)

In [ ]:
#we will store this result as base_acc to use it later
base_acc = accuracy_score(train_targets, train_preds), model.score(X_val, val_targets)
base_acc

#### Visualization of Decision Tree

In [ ]:
from sklearn.tree import plot_tree, export_text

In [ ]:
plt.figure(figsize=(80,20))
plot_tree(model, feature_names=X_train.columns, max_depth=2, filled=True);

In [ ]:
model.tree_.max_depth

In [ ]:
tree_text = export_text(model, max_depth=10, feature_names=list(X_train.columns))
print(tree_text[:5000])

#### Feature Importance

In [ ]:
model.feature_importances_

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
importance_df.head(10)

In [ ]:
plt.title('Feature Importance')
sns.barplot(data=importance_df.head(10), x='importance', y='feature');

#### Hyperparameter Tuning
We will define a function that will check the accuracy with different parameters and their various values over a range. Also plotting the graph of relation between the accuracy and parameter values.

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(random_state=42, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
def test_param_and_plot(param_name, param_values):
    train_acc, val_acc = [], [] 
    for value in param_values:
        params = {param_name: value}
        train_score, val_score = test_params(**params)
        train_acc.append(train_score)
        val_acc.append(val_score)
    plt.figure(figsize=(10,6))
    plt.title('Overfitting curve: ' + param_name)
    plt.plot(param_values, train_acc)
    plt.plot(param_values, val_acc)
    plt.xlabel(param_name)
    plt.ylabel('Accuracy')
    plt.legend(['Training', 'Validation'])
    print('Max Acc by:', val_acc.index(max(val_acc))+ int(min(param_values)))

#### `criterion`

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='gini', random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

`criterion='entropy'` increases the accuracy

#### `splitter`

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', splitter='random', random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', splitter='best', random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

though `splitter='best'` gives more accuracy than the `splitter='random'` but it is same as before cause it is a default arguement

#### `max_depth`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(random_state=42,criterion='gini', **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_depth=[i for i in range(1,32)]
test_param_and_plot('max_depth',max_depth)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `max_leaf_nodes`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(criterion='entropy',max_depth=6, random_state=42, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_leaf_nodes=[i for i in range(2,150)]
test_param_and_plot('max_leaf_nodes',max_leaf_nodes)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_samples_split`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                                   random_state=42, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_samples_split=[i for i in range(2,300)]
test_param_and_plot('min_samples_split',min_samples_split)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_samples_leaf`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                                   min_samples_split=2, random_state=42, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_samples_leaf=[i for i in range(1,150)]
test_param_and_plot('min_samples_leaf',min_samples_leaf)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `max_features`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_features=[i for i in range(1,11)]
test_param_and_plot('max_features',max_features)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_impurity_decrease`

In [ ]:
def test_params(**params):
    model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_impurity_decrease=[1.0e-10,0, 1.0e-9, 1.0e-7, 1.0e-5, 1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0]
test_param_and_plot('min_impurity_decrease',min_impurity_decrease)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0)
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `class_weight`

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0,class_weight={0:15.5, 1:1})
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0,class_weight={0:1.5, 1:1})
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0,class_weight={0:5, 1:1})
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0,class_weight={0:1, 1:0})
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0,class_weight={0:1, 1:0.5})
model.fit(X_train, train_targets)
print(base_acc)
model.score(X_train, train_targets), model.score(X_val, val_targets)

None of the combination of weights are seem to be working for further increment in the accuracy.

### Predictions with Best Parameter Values

In [ ]:
model = DecisionTreeClassifier(criterion='entropy',max_depth=6, max_leaf_nodes=27,
                               min_samples_split=2, random_state=42,
                               min_samples_leaf=1,
                               max_features=10,
                               min_impurity_decrease=0.0)
model.fit(X_train, train_targets)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
test_preds=model.predict(test_inputs)
test_preds

In [ ]:
pd.value_counts(test_preds) / len(test_preds)

In [ ]:
submission['Probability']=test_preds
submission.head()

In [ ]:
#Saving Submissions as CSV File
submission.to_csv('submission.csv', index=None)

It gives the accuracy of 77.89% after submitting to the competition, which is good but we will have to try different algorithm.

### Modeling : RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(n_jobs=-1, random_state=42)

In [ ]:
model.fit(X_train, train_targets)

In [ ]:
model.score(X_train, train_targets)

In [ ]:
model.score(X_val, val_targets)

Accuracy on validation data without hyper parameter tuning is significantly more than that of the untuned DecisionTreeClassifier. Since RandomForestClassifier uses multiple decision trees (n_estimators=100 by default) to make predictions and gives the prediction which the average of all the predictions given by each tree. We can see that none of the tree has accuracy higher than the decision tree.

In [ ]:
len(model.estimators_)

In [ ]:
def estimator_acc(i):
    model.estimators_[i].fit(X_train, train_targets)
    return model.estimators_[i].score(X_val, val_targets)

In [ ]:
accuracy=[]
for i in range(0,100):
    accuracy.append(estimator_acc(i))

In [ ]:
max(accuracy)

See here the maximum accuracy of individual tree is `71.48%`  where as accuracy of RandomForest is `78.16%`

In [ ]:
train_probs = model.predict_proba(X_train)
train_probs

#### Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
importance_df.head(10)

In [ ]:
plt.title('Feature Importance')
sns.barplot(data=importance_df.head(10), x='importance', y='feature');

#### Hyperparameter Tuning
We will define a function that will check the accuracy with different parameters and their various values over a range. Also plotting the graph of relation between the accuracy and parameter values.

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(n_jobs=-1, random_state=42, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
def test_param_and_plot(param_name, param_values):
    train_acc, val_acc = [], [] 
    for value in param_values:
        params = {param_name: value}
        train_score, val_score = test_params(**params)
        train_acc.append(train_score)
        val_acc.append(val_score)
    plt.figure(figsize=(10,6))
    plt.title('Overfitting curve: ' + param_name)
    plt.plot(param_values, train_acc)
    plt.plot(param_values, val_acc)
    plt.xlabel(param_name)
    plt.ylabel('Accuracy')
    plt.legend(['Training', 'Validation'])
    print('Max Acc by:', val_acc.index(max(val_acc))+ int(min(param_values)))

In [ ]:
base_model = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train, train_targets)
base_train_acc = base_model.score(X_train, train_targets)
base_val_acc = base_model.score(X_val, val_targets)
base_accs = base_train_acc, base_val_acc
base_accs

#### `n_estimator`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
n_estimators=[100, 200, 300, 400, 500,600, 700, 800, 900, 1000]
test_param_and_plot('n_estimators',n_estimators)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=800)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `max_depth`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=800, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_depth=[i for i in range(1,32)]
test_param_and_plot('max_depth',max_depth)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=800, max_depth=12)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `max_leaf_nodes`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,
                                   n_estimators=800, max_depth=12, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_leaf_nodes=[i for i in range(10,200)]
test_param_and_plot('max_leaf_nodes',max_leaf_nodes)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `max_features`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
max_features=[i for i in range(1,11)]
test_param_and_plot('max_features',max_features)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_samples_split`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                                   **params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_samples_split=[i for i in range(10,40)]
test_param_and_plot('min_samples_split',min_samples_split)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_samples_leaf`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,**params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_samples_leaf=[i for i in range(1,150)]
test_param_and_plot('min_samples_leaf',min_samples_leaf)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `min_impurity_decrease`

In [ ]:
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,**params)
    model.fit(X_train, train_targets)
    train_score = accuracy_score(model.predict(X_train), train_targets)
    val_score = accuracy_score(model.predict(X_val), val_targets)
    return train_score, val_score
min_impurity_decrease=[1.0e-10,0, 1.0e-9, 1.0e-7, 1.0e-5, 1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0]
test_param_and_plot('min_impurity_decrease',min_impurity_decrease)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

#### `bootstrap` 
No need to change the default boolean value which is `True`, since model performs better with bootstrap enabled.

#### `class_weight`

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0, class_weight={0:4, 1:1})
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0, class_weight={0:1.5, 1:1})
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0, class_weight={0:1, 1:0})
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

None of the combinations of weights seem to be working here

### Predictions with Best Parameter Values

In [ ]:
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0)
model.fit(X_train, train_targets)
print(base_accs)
model.score(X_train, train_targets), model.score(X_val, val_targets)

In [ ]:
test_preds=model.predict(test_inputs)
test_preds

In [ ]:
pd.value_counts(test_preds) / len(test_preds)

In [ ]:
submission['Probability']=test_preds
submission.head()

In [ ]:
#Saving the submissions
submission.to_csv('submission1.csv', index=None)

It gives the accuracy of 78.56% which is greater than what DecisionTreeClassifier's output had given.

### Saving the Model

In [ ]:
import joblib

In [ ]:
financial_distress = {
    'model': model,
    'imputer': imputer,
    'scaler': scaler,
    'input_cols': input_cols,
    'target_col': target_col
}
#all input calls are numeric cols

In [ ]:
joblib.dump(financial_distress, 'financial_distress')

In [ ]:
financial_distress2 = joblib.load('financial_distress')

In [ ]:
val_preds2 = financial_distress2['model'].predict(X_val)
accuracy_score(val_targets, val_preds2)

We can use this saved model anywhere we want.

<div class="alert alert-block alert-info"> <h2>📌Summary :</h2><br>
    <p style='color:black;'>
        I created this project for Machine Learning with Python: Zero to GBMs offered by Jovian.ai.<br><br>
        <b>Under this project,</b><br>
        1. I learned to extract and prepare data for training.<br>
        2. Performed exploratory data analysis to make sense of the data and to observe its distribution.<br>
        3. I also scaled the data to bring it down to considerable ranges, and filled the missing values using `SimpleImputer`.<br>
        4. Trained two models using DecisionTreeClassifier and RandomForestClassifier.<br>
        5. Visualized tree using plot_tree and export_text.<br>
        6. Tuned many hyperparameters to achieve higher accuracy.<br>
        7. Made predictions on the test data and submitted them to the competition where I achieved 72% accuracy with DecisionTreeClassifier and 70% accuaracy with RandomForestClassifier().<br>
    </p>
</div>

## Future Work
My aim is to achieve highest accuracy for the predictions on the given dataset, therefore I'm planning to learn and use `Deep Learning Algorithms` after this course. I will also use this saved model from joblib to create a web application which will take inputs from the user and give output in the form of whether somebody is likely to suffer financial distress or not in near future. I will make its interefece as user-frinedly as possible.

## References
1. [Data-Manipulation](https://www.geeksforgeeks.org/data-manipulattion-in-python-using-pandas/)
1. [Kaggle](https://www.kaggle.com/)
1. [Scikit-learn](https://scikit-learn.org/stable/index.html) 
2. [Decision-Tree](https://www.geeksforgeeks.org/decision-tree/)
3. [Random-Forest](https://machinelearningknowledge.ai/python-sklearn-random-forest-classifier-tutorial-with-example/)
4. [Hyperparameter-Tuning](https://towardsdatascience.com/how-to-tune-hyperparameters-of-machine-learning-models-a82589d48fc8)

##### The End